# Знакомство с Ultralytics и детекция людей на MOT17

## 0 Окружение

Ultralytics объединяет несколько **задач** — detection, segmentation, classification, pose и oriented boxes — и несколько **режимов работы**: `train`, `val`, `predict`, `track`, `export`, `benchmark`.

Нам нужны два режима:

- `predict` независимо находит людей на каждом кадре;
- `track` добавляет постоянный ID, связывая детекции во времени.

Если импорт не работает, один раз выполните отдельную ячейку `%pip install -U ultralytics`, затем перезапустите kernel. Не помещайте установку пакетов в цикл обработки кадров.

Документация: [Predict](https://docs.ultralytics.com/modes/predict/), [Track](https://docs.ultralytics.com/modes/track/), [Train](https://docs.ultralytics.com/modes/train/).

In [ ]:
!pip install -U ultralytics

In [ ]:
# Раскомментируйте только при отсутствии пакета, затем перезапустите kernel.
# %pip install -U ultralytics

from pathlib import Path
from collections import defaultdict
import configparser
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy.optimize import linear_sum_assignment

try:
    import torch
    import ultralytics
    from ultralytics import YOLO
except ImportError as error:
    raise ImportError("Установите пакет: %pip install -U ultralytics") from error

print("Python:", platform.python_version())
print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Устройство:", DEVICE)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Проверяем скачанные данные

https://motchallenge.net/data/MOT17/

Из архива используется только `MOT17-04-FRCNN`: 1050 кадров 1920×1080, `gt/gt.txt`, публичные FRCNN-детекции и `seqinfo.ini`. Суффикс `FRCNN` относится к готовому файлу `det/det.txt`; сами изображения и GT у вариантов DPM/FRCNN/SDP одной исходной сцены совпадают.

GT имеет строки

`frame,id,left,top,width,height,mark,class,visibility`

Для нашего визуального baseline оставляем `mark=1` и `class=1` — размеченных пешеходов.

In [ ]:
import re

def find_sequence():
    candidates = [
        Path("./data/MOT17/train/MOT17-04-FRCNN"),
    ]
    for candidate in candidates:
        if (candidate / "seqinfo.ini").exists():
            return candidate
    raise FileNotFoundError("Не найдена MOT17-04-FRCNN. Проверьте папку 11/data/MOT17/train.")


SEQ_DIR = find_sequence()
IMG_DIR = SEQ_DIR / "img1"
GT_PATH = SEQ_DIR / "gt" / "gt.txt"

cfg = configparser.ConfigParser(); cfg.read(SEQ_DIR / "seqinfo.ini")
SEQ_LENGTH = int(cfg["Sequence"]["seqLength"])
WIDTH = int(cfg["Sequence"]["imWidth"]); HEIGHT = int(cfg["Sequence"]["imHeight"])
FRAME_RATE = int(cfg["Sequence"]["frameRate"])

# Фильтруем только оригинальные файлы кадров без дубликатов вида '000221 (1).jpg'
all_jpgs = IMG_DIR.glob("*.jpg")
frame_paths = []
for path in all_jpgs:
    # Проверяем, что имя файла состоит ровно из 6 цифр (классический формат кадров MOT17)
    if re.match(r'^\d{6}$', path.stem):
        frame_paths.append(path)

frame_paths = sorted(frame_paths, key=lambda path: int(path.stem))

assert len(frame_paths) == SEQ_LENGTH == 1050, f"Найдено кадров: {len(frame_paths)} вместо {SEQ_LENGTH}"
assert GT_PATH.exists()
print(f"{SEQ_DIR.name}: {SEQ_LENGTH} кадров, {WIDTH}×{HEIGHT}, {FRAME_RATE} FPS")
print("Первый кадр:", frame_paths[0])

In [ ]:
def load_mot_gt(path):
    rows = np.loadtxt(path, delimiter=",", ndmin=2)
    pedestrian = rows[(rows[:, 6] == 1) & (rows[:, 7] == 1)]
    by_frame = defaultdict(list)
    for frame, object_id, left, top, width, height, mark, cls, visibility in pedestrian:
        by_frame[int(frame)].append(
            [int(object_id), left, top, left+width, top+height, visibility]
        )
    return {frame: np.asarray(items, dtype=float).reshape(-1, 6)
            for frame, items in by_frame.items()}


GT = load_mot_gt(GT_PATH)
gt_counts = pd.Series({frame: len(rows) for frame, rows in GT.items()})
SAMPLE_FRAME = int(gt_counts.loc[gt_counts.index <= 120].idxmax())
print("GT pedestrian boxes:", int(gt_counts.sum()))
print("Кадр для демонстрации:", SAMPLE_FRAME, "людей:", int(gt_counts[SAMPLE_FRAME]))

In [ ]:
image = plt.imread(IMG_DIR / f"{SAMPLE_FRAME:06d}.jpg")
plt.figure(figsize=(14, 7)); plt.imshow(image); plt.axis("off")
for object_id, x1, y1, x2, y2, visibility in GT[SAMPLE_FRAME]:
    color = "lime" if visibility >= 0.5 else "yellow"
    plt.gca().add_patch(Rectangle((x1, y1), x2-x1, y2-y1,
                                  fill=False, edgecolor=color, linewidth=1.2))
plt.title(f"GT кадр {SAMPLE_FRAME}: зелёный visibility≥0.5, жёлтый <0.5")
plt.show()

## 1 Модель и один кадр

`YOLO("yolo11n.pt")` создаёт единый объект модели. Суффикс `n` означает nano: минимальная задержка и память, но ниже качество на маленьких и закрытых людях.

Вызов модели возвращает список `Results`, по одному элементу на изображение. Для detection наиболее важны:

- `result.boxes.xyxy` — координаты `(x1,y1,x2,y2)`;
- `result.boxes.conf` — confidence;
- `result.boxes.cls` — индекс класса;
- `result.names` — отображение индекса в название;
- `result.speed` — время preprocess, inference и postprocess;
- `result.plot()` — готовая BGR-визуализация.

In [ ]:
local_weights = [Path("models/yolo11n.pt"), Path("11/models/yolo11n.pt")]
WEIGHTS = str(next((path for path in local_weights if path.exists()), Path("yolo11n.pt")))
model = YOLO(WEIGHTS)
print("Задача модели:", model.task)
print("Класс 0:", model.names[0])

result = model.predict(
    source=str(IMG_DIR / f"{SAMPLE_FRAME:06d}.jpg"),
    conf=0.10, imgsz=960, classes=[0], device=DEVICE, verbose=False,
)[0]

print("Число people detections:", len(result.boxes))
print("Формы xyxy/conf/cls:", result.boxes.xyxy.shape,
      result.boxes.conf.shape, result.boxes.cls.shape)
print("Скорость, ms:", result.speed)
print(result.boxes.data[:5].cpu().numpy())

In [ ]:
# result.plot() возвращает BGR-массив, поэтому меняем порядок каналов для matplotlib.
annotated_rgb = result.plot()[:, :, ::-1]
plt.figure(figsize=(14, 7)); plt.imshow(annotated_rgb); plt.axis("off")
plt.title(f"{WEIGHTS}, conf=0.10, imgsz=960")
plt.show()

### Мини эксперимент

1. Что сильнее изменит число найденных людей: `conf` или `imgsz`?
2. Где появятся дополнительные боксы при уменьшении `conf`: на хорошо видимых людях или на частично закрытых?
3. Почему нельзя выбирать порог только по красивой картинке одного кадра?

`conf` фильтрует уже найденные кандидаты. `imgsz` меняет размер входа сети: крупный размер помогает мелким объектам, но увеличивает вычисления примерно квадратично по стороне.

In [ ]:
experiments = [(0.50, 640), (0.10, 640), (0.10, 1280)]
fig, axes = plt.subplots(1, 3, figsize=(19, 6))
for ax, (conf, imgsz) in zip(axes, experiments):
    pred = model.predict(str(IMG_DIR / f"{SAMPLE_FRAME:06d}.jpg"),
                         conf=conf, imgsz=imgsz, classes=[0],
                         device=DEVICE, verbose=False)[0]
    ax.imshow(pred.plot()[:, :, ::-1]); ax.axis("off")
    ax.set_title(f"conf={conf}, imgsz={imgsz}, boxes={len(pred.boxes)}")
plt.tight_layout()

## 2 Потоковая обработка и кэш

Для последовательности используем `stream=True`: результаты приходят по одному, а не накапливаются вместе с изображениями в памяти. Кэшируем **сырые детекции**, а не только картинку с подписями. Тогда порог можно менять и трекеры можно сравнивать без повторного запуска нейросети.

Сохраняем `conf≥0.05`, а учебный baseline оцениваем при `0.10`. Более низкий порог кэша позволяет построить зависимость качества от `conf`. Ключ кэша включает модель, `imgsz` и число кадров: старые результаты нельзя молча выдавать за новый эксперимент.

In [ ]:
N_FRAMES = 120      # None -> все 1050 кадров
PREDICT_CONF = 0.05
PREDICT_IMGSZ = 960

selected_paths = frame_paths if N_FRAMES is None else frame_paths[:N_FRAMES]
ARTIFACT_DIR = SEQ_DIR.parents[3] / "artifacts" / "ultralytics_mot17"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
limit_tag = "all" if N_FRAMES is None else str(N_FRAMES)
CACHE_PATH = ARTIFACT_DIR / f"{SEQ_DIR.name}_{Path(WEIGHTS).stem}_{PREDICT_IMGSZ}_{limit_tag}.csv"


def predict_to_cache(model, paths, cache_path, conf=0.05, imgsz=960):
    if cache_path.exists():
        print("Загружаем кэш:", cache_path)
        return pd.read_csv(cache_path)

    stream = model.predict(source=[str(path) for path in paths], stream=True,
                           conf=conf, imgsz=imgsz, classes=[0],
                           device=DEVICE, verbose=False)
    records = []
    for path, prediction in zip(paths, stream):
        frame = int(path.stem)
        if prediction.boxes is None or len(prediction.boxes) == 0:
            continue
        xyxy = prediction.boxes.xyxy.cpu().numpy()
        scores = prediction.boxes.conf.cpu().numpy()
        classes = prediction.boxes.cls.cpu().numpy().astype(int)
        for box, score, cls in zip(xyxy, scores, classes):
            records.append([frame, *box, float(score), int(cls)])
    table = pd.DataFrame(records, columns=["frame", "x1", "y1", "x2", "y2", "score", "class"])
    table.to_csv(cache_path, index=False)
    print("Сохранено:", cache_path, "строк:", len(table))
    return table


DETECTIONS = predict_to_cache(model, selected_paths, CACHE_PATH,
                              conf=PREDICT_CONF, imgsz=PREDICT_IMGSZ)
display(DETECTIONS.head())
print(DETECTIONS["score"].describe().round(3))

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(DETECTIONS["score"], bins=np.linspace(0, 1, 20), color="tab:blue", alpha=0.8)
for threshold, color in [(0.10, "tab:orange"), (0.25, "tab:red"), (0.50, "black")]:
    plt.axvline(threshold, color=color, linestyle="--", label=f"conf={threshold}")
plt.xlabel("confidence"); plt.ylabel("число боксов"); plt.title("Распределение score")
plt.legend(); plt.show()

## 3 Сравнение YOLO с GT

Сопоставляем предсказания и GT внутри каждого кадра венгерским алгоритмом по стоимости `1-IoU`. Пара считается истинно положительной при `IoU≥0.5`.

- `TP` — предсказание получило уникальный GT;
- `FP` — предсказание не получило GT;
- `FN` — GT остался без предсказания;
- `precision = TP/(TP+FP)`;
- `recall = TP/(TP+FN)`;
- `F1` — гармоническое среднее precision и recall.

Это оценка **детектора**, не трекера: ID пока не учитываются. Для полной MOT-оценки нужны последовательные ID и TrackEval.

In [ ]:
def iou_matrix(a, b):
    a = np.asarray(a, dtype=float).reshape(-1, 4)
    b = np.asarray(b, dtype=float).reshape(-1, 4)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb-lt, 0, None)
    inter = wh[..., 0] * wh[..., 1]
    area_a = (a[:, 2]-a[:, 0]) * (a[:, 3]-a[:, 1])
    area_b = (b[:, 2]-b[:, 0]) * (b[:, 3]-b[:, 1])
    union = area_a[:, None] + area_b[None, :] - inter
    return np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)


def detection_metrics(detections, gt, frames, conf=0.10, iou_threshold=0.5):
    tp = fp = fn = 0
    for frame in frames:
        gt_boxes = gt.get(frame, np.empty((0, 6)))[:, 1:5]
        pred_boxes = detections.query("frame == @frame and score >= @conf")[["x1", "y1", "x2", "y2"]].to_numpy()
        if len(gt_boxes) and len(pred_boxes):
            ious = iou_matrix(gt_boxes, pred_boxes)
            rows, cols = linear_sum_assignment(1-ious)
            matched = sum(ious[r, c] >= iou_threshold for r, c in zip(rows, cols))
        else:
            matched = 0
        tp += matched; fn += len(gt_boxes)-matched; fp += len(pred_boxes)-matched
    precision = tp/max(tp+fp, 1); recall = tp/max(tp+fn, 1)
    f1 = 2*precision*recall/max(precision+recall, 1e-12)
    return {"conf": conf, "TP": tp, "FP": fp, "FN": fn,
            "precision": precision, "recall": recall, "F1": f1}


evaluated_frames = [int(path.stem) for path in selected_paths]
thresholds = [0.05, 0.10, 0.20, 0.30, 0.50, 0.70]
metrics_table = pd.DataFrame([
    detection_metrics(DETECTIONS, GT, evaluated_frames, conf=value)
    for value in thresholds
])
display(metrics_table.round(3))
best_row = metrics_table.iloc[metrics_table["F1"].argmax()]
print(f"Лучший F1 на smoke-test: conf={best_row['conf']:.2f}, F1={best_row['F1']:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(metrics_table["conf"], metrics_table["precision"], "o-", label="precision")
ax.plot(metrics_table["conf"], metrics_table["recall"], "o-", label="recall")
ax.plot(metrics_table["conf"], metrics_table["F1"], "o-", label="F1")
ax.set(xlabel="conf", ylabel="метрика", ylim=(0, 1.02), title="Порог confidence: компромисс FP и FN")
ax.legend(); plt.show()

In [ ]:
def show_gt_vs_prediction(frame, conf=0.10):
    image = plt.imread(IMG_DIR / f"{frame:06d}.jpg")
    fig, ax = plt.subplots(figsize=(15, 8)); ax.imshow(image); ax.axis("off")
    for object_id, x1, y1, x2, y2, visibility in GT.get(frame, np.empty((0, 6))):
        ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False,
                               edgecolor="lime", linestyle="--", linewidth=1.2))
    selected = DETECTIONS.query("frame == @frame and score >= @conf")
    for row in selected.itertuples():
        color = "cyan" if row.score >= 0.6 else "orange"
        ax.add_patch(Rectangle((row.x1, row.y1), row.x2-row.x1, row.y2-row.y1,
                               fill=False, edgecolor=color, linewidth=1.2))
    ax.set_title(f"Кадр {frame}: GT — зелёный пунктир; YOLO — cyan/orange; conf≥{conf}")
    plt.show()


show_gt_vs_prediction(SAMPLE_FRAME, conf=0.10)


1. Посмотрите на людей с низкой `visibility`. Чаще ли они становятся FN?
2. Найдите FP. Это действительно «ошибка», или GT исключил человека из оцениваемого класса?
3. Почему лучший `conf` на первых 120 кадрах нельзя автоматически объявить лучшим для всей MOT17?
4. Что изменится при переходе с `yolo11n.pt` на `yolo11m.pt`: precision, recall, скорость, память?


## 4 Track mode и ByteTrack

`model.track(..., persist=True)` сообщает трекеру, что следующий вызов продолжает ту же последовательность. `boxes.id` появляется только у результатов, которым трекер назначил ID.

In [ ]:
TRACK_FRAMES = min(60, len(selected_paths))
PREVIEW_COUNT = 6
TRACK_PATH = ARTIFACT_DIR / f"{SEQ_DIR.name}_bytetrack_first{TRACK_FRAMES}.txt"


def run_bytetrack(model, paths, output_path, conf=0.10, imgsz=960,
                  preview_count=6):
    rows, previews = [], []
    for path in paths:
        result = model.track(
            source=str(path), persist=True, tracker="bytetrack.yaml",
            conf=conf, imgsz=imgsz, classes=[0], device=DEVICE, verbose=False,
        )[0]
        frame = int(path.stem)

        # Храним только последние preview_count изображений, а не всё видео в RAM.
        previews.append((frame, result.plot()[:, :, ::-1].copy()))
        previews = previews[-preview_count:]

        if result.boxes is not None and result.boxes.id is not None:
            xyxy = result.boxes.xyxy.cpu().numpy()
            ids = result.boxes.id.int().cpu().numpy()
            scores = result.boxes.conf.cpu().numpy()
            for (x1, y1, x2, y2), track_id, score in zip(xyxy, ids, scores):
                rows.append([frame, int(track_id), x1, y1, x2-x1, y2-y1,
                             score, -1, -1, -1])
    table = np.asarray(rows, dtype=float).reshape(-1, 10)
    np.savetxt(output_path, table, delimiter=",",
               fmt=["%d", "%d", "%.3f", "%.3f", "%.3f", "%.3f", "%.4f", "%d", "%d", "%d"])
    return table, previews


track_rows, track_previews = run_bytetrack(
    model, selected_paths[:TRACK_FRAMES], TRACK_PATH,
    conf=0.10, imgsz=PREDICT_IMGSZ, preview_count=PREVIEW_COUNT,
)
print("Строк треков:", len(track_rows), "уникальных ID:", len(np.unique(track_rows[:, 1])))
print("MOTChallenge-файл:", TRACK_PATH)

In [ ]:
if track_previews:
    ncols = 3
    nrows = int(np.ceil(len(track_previews) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5.5*nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, (frame, annotated_rgb) in zip(axes, track_previews):
        ax.imshow(annotated_rgb)
        ax.set_title(f"ByteTrack, кадр {frame}")
        ax.axis("off")
    for ax in axes[len(track_previews):]:
        ax.axis("off")
    plt.suptitle("Последовательные кадры: проверьте постоянство ID", fontsize=15)
    plt.tight_layout()
    plt.show()